# Cretona — Kaggle Wan2.1 worker

Truoc khi Run All:

1. **Settings -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On**
3. **Add-ons -> Secrets**: bat du 5 secret (xem cell duoi)
4. **Add Input**: upload `cretona-kaggle-worker-AUDIO.zip` lam private dataset

Notebook nay KHONG chua bat ky credential nao. Moi bi mat lay tu Kaggle Secrets.


## 1. Nap 5 Kaggle Secrets

| Secret | Gia tri |
|---|---|
| `CRETONA_WORKER_DATABASE_URL` | Session-pooler PostgreSQL URL cua `cretona_kaggle_worker`, co `sslmode=require` |
| `CRETONA_SUPABASE_URL` | `https://<project-ref>.supabase.co` |
| `CRETONA_SUPABASE_PUBLISHABLE_KEY` | Publishable key cua project (anon key cu cung duoc) |
| `CRETONA_WORKER_EMAIL` | Email tai khoan Auth danh rieng cho worker |
| `CRETONA_WORKER_PASSWORD` | Mat khau tai khoan do |

**Khong bao gio** them `service_role` hay URL admin cua backend vao Kaggle.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
for name in (
    "CRETONA_WORKER_DATABASE_URL",
    "CRETONA_SUPABASE_URL",
    "CRETONA_SUPABASE_PUBLISHABLE_KEY",
    "CRETONA_WORKER_EMAIL",
    "CRETONA_WORKER_PASSWORD",
):
    os.environ[name] = secrets.get_secret(name)

print("da nap", 5, "secret (khong in gia tri)")

# Two real worker jobs shipped an audible AAC track on 2026-09-03; see
# docs/evidence/EXP006_FIRST_AUDIO_JOBS_2026-09-03.md. This flag only works
# with the audio-capable bundle below - the silent decc5f5d worker ignores it
# entirely, so the flag and the hash must move together. Not a credential.
ENABLE_AUDIO = True
os.environ["CRETONA_WORKER_AUDIO"] = "1" if ENABLE_AUDIO else "0"
print("audio:", os.environ["CRETONA_WORKER_AUDIO"])


## 2. Xac minh va copy source tu private dataset

Hash duoi day tinh tren dang **LF** (blob git), khong phai file CRLF tren Windows.
Ban demo 2026-09-05 dung bundle audio `04d3e70b...`, da chay live 2 job that ngay 2026-09-03. Khong doi dataset hoac hash truoc buoi demo.


In [ ]:
import hashlib
import shutil
from pathlib import Path

expected = {
    "backend/kaggle_worker.py":
        "04d3e70b78cd8a90c52644ac7d69e45ce75c58a98c95d52345bc0a9ad47c82ea",
    "backend/requirements-kaggle.txt":
        "46263bf0527742dde0d98a5da163afc1646a53b5f6de545ad3e94da43c99c669",
}
root = Path("/kaggle/working/Cretona")
for relative, expected_sha256 in expected.items():
    candidates = list(Path("/kaggle/input").rglob(relative))
    matches = [
        path for path in candidates
        if hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha256
    ]
    assert len(matches) == 1, {
        "relative": relative,
        "candidateCount": len(candidates),
        "matchingCount": len(matches),
    }
    source = matches[0]
    actual = hashlib.sha256(source.read_bytes()).hexdigest()
    assert actual == expected_sha256, f'{relative}: {actual}'
    destination = root / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print("OK", relative)


## 3. Smoke test re tien (khong nap model)

Hang doi rong thi thoat 0 va khong tai model. Chay de kiem tra Auth + ket noi
database truoc khi ton thoi gian cai dat.


In [ ]:
%cd /kaggle/working/Cretona
!python -m pip install -q "psycopg[binary]==3.3.4"
!python backend/kaggle_worker.py --once


## 4. Cai day du dependency

**Khong** cai hay nang cap `torch` — requirements co y dung ban CUDA Kaggle da co.


In [ ]:
!python -m pip install -q -r backend/requirements-kaggle.txt


## 4b. Cai MMAudio (chi khi bat audio)

Worker tim MMAudio o `/kaggle/working/MMAudio`. **Thieu buoc nay thi audio
khong bao gio chay** — `add_audio` chi ghi log rồi tra ve video im lang, job
van hoan tat binh thuong.

Bo qua cell nay neu khong bat `CRETONA_WORKER_AUDIO`.

> Checkpoint MMAudio la **CC-BY-NC 4.0**: dung cho do an duoc, thuong mai thi khong.

> Lan chay dau tien se tai ~6 GB weight, nen **job dau tien co audio lau hon
> khoang 3 phut** so voi cac job sau.


In [ ]:
import os, subprocess, sys, torch
from pathlib import Path

mmaudio = Path('/kaggle/working/MMAudio')
if os.environ.get('CRETONA_WORKER_AUDIO') == '1':
    if not mmaudio.is_dir():
        subprocess.run(
            ['git', 'clone', '-q', 'https://github.com/hkchengrex/MMAudio.git', str(mmaudio)],
            check=True,
        )
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(mmaudio)], check=True)
    print('torch      :', torch.__version__)
    print('demo.py co :', (mmaudio / 'demo.py').is_file())
else:
    print('audio disabled: bo qua tai va cai MMAudio')


## 5. Chay worker

Cell nay chay lien tuc va phuc vu ca hang doi. De nguyen trong suot buoi demo.

- 25 denoising steps, 16 fps, co audio (`CRETONA_WORKER_AUDIO=1`)
- Clip 4 giay = 65 frame. Hai job that ngay 2026-09-03 mat **27,7 va 29,6 phut**
  end to end, da tinh ca audio
- Clip co audio nang 2,9-5,1 MB thay vi ~0,4 MB: MMAudio encode lai video
- Chi xu ly 1 job mot luc

Muon chi chay dung 1 job roi dung thi them `--once`.


In [ ]:
!python backend/kaggle_worker.py
